In [4]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import f_classif
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.formula.api import ols
import numpy as np
from tqdm import tqdm
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests
import os


### Feautre Selection

In [23]:
# Step 1: Load features
df_features = pd.read_csv("data/All_features_corrected_final_patient.csv").reset_index(drop=True)

# Step 2: Load and concat metadata files in correct order
folder = "data/ModelsUncorrected"
csv_files = sorted([f for f in os.listdir(folder) if f.endswith(".csv")])

metadata_list = []
for f in csv_files:
    df = pd.read_csv(os.path.join(folder, f), usecols=["PatientName", "Model", "Wavelength", "GLbins"])
    df["Reconstruction"] = "MB" if "_MB" in f else "BP"
    metadata_list.append(df)

df_metadata = pd.concat(metadata_list, ignore_index=True)
df_metadata = df_metadata.reset_index(drop=True)

# Step 3: Verify same number of rows
assert len(df_features) == len(df_metadata) == 2268, "Row count mismatch"

# Step 4: Concatenate by row index
df_combined = pd.concat([df_features, df_metadata], axis=1)

# Step 5: Convert to category
for col in ["Model", "Wavelength", "GLbins", "Reconstruction"]:
    df_combined[col] = df_combined[col].astype("category")

print("✅ Final combined shape:", df_combined.shape)
df_combined


# Find all columns including their actual name object (even if repeated)
cols = pd.Series(df_combined.columns)
duplicates = cols[cols.duplicated(keep=False)]
# print("Duplicate columns:\n", duplicates)

df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()]


# Get all luminal patient IDs
luminal_ids = df_combined[df_combined["Model"] == "luminal"]["PatientName"].unique()

# Pick one randomly (or use a fixed ID for reproducibility)
np.random.seed(42)
drop_id = np.random.choice(luminal_ids, 1)[0]
print(f"Dropping Luminal patient: {drop_id}")

# Drop all rows for that patient
df_combined = df_combined[df_combined["PatientName"] != drop_id].reset_index(drop=True)
print("✅ Final combined shape after dropping patient:", df_combined.shape)


✅ Final combined shape: (2268, 99)
Dropping Luminal patient: 198
✅ Final combined shape after dropping patient: (2160, 98)


In [24]:
df_combined

,PatientName,FO 10Percentile,FO 90Percentile,FO Energy,FO Entropy,FO InterquartileRange,FO Kurtosis,FO Maximum,FO MeanAbsoluteDeviation,FO Mean,...,GLSZM ZoneVariance,NGTDM Busyness,NGTDM Coarseness,NGTDM Complexity,NGTDM Contrast,NGTDM Strength,GLbins,Model,Wavelength,Reconstruction
0,10,69.142880,290.878683,1.925446e+06,-211066.055663,99.644165,23.988499,1506.465942,80.712806,167.184233,...,0.005226,141.041268,7.021165e+05,5.431388,-0.008386,-1.346671e+15,128,basal,700,BP
1,10,68.959734,246.576346,1.913616e+06,-213433.597164,80.613131,24.513340,1244.968140,64.953736,148.365253,...,0.003608,137.561953,6.350452e+05,5.512007,-0.008424,-1.347603e+15,128,basal,730,BP
2,10,70.308770,265.578290,1.918571e+06,-214352.172688,87.411236,24.715534,1360.987915,71.373395,156.751060,...,0.005046,140.418469,7.751153e+05,5.212363,-0.008415,-1.346064e+15,128,basal,750,BP
3,10,69.735306,273.831262,1.920929e+06,-210314.678012,91.426805,24.677900,1426.273926,74.664145,160.247312,...,0.003304,141.307182,5.620439e+05,5.830865,-0.008369,-1.347809e+15,128,basal,760,BP
4,10,68.767374,252.913724,1.915210e+06,-210869.408803,83.389416,24.597626,1282.152344,67.249362,150.974229,...,0.003673,140.365238,6.884752e+05,5.429985,-0.008394,-1.346774e+15,128,basal,770,BP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2155,98,992.082990,4457.898047,9.779358e+06,314845.586861,1563.086426,9.743394,18382.693359,1110.867556,2444.303064,...,0.000031,96.333320,9.712949e+05,82287.526364,-0.001564,1.369483e+16,8,basal,770,MB
2156,98,851.154541,4067.293140,8.219187e+06,347907.061977,1421.651093,7.998629,14729.750000,1009.281441,2205.240598,...,0.000031,96.332213,1.128664e+06,105620.991383,0.000838,1.634342e+16,8,basal,800,MB
2157,98,838.989789,4029.269922,8.086392e+06,353231.076936,1419.266876,7.754886,14415.465820,1001.488346,2183.198058,...,0.000031,96.335268,1.169228e+06,104233.632403,0.001966,1.582569e+16,8,basal,820,MB
2158,98,878.749854,4083.819263,8.325601e+06,363272.297324,1443.129120,7.466826,13780.784180,1010.279500,2232.959583,...,0.000031,96.333182,1.179483e+06,110186.114695,0.003280,1.493451e+16,8,basal,840,MB


In [25]:
# Define numeric features
feature_cols = df_combined.drop(columns=["PatientName", "Model", "Wavelength", "GLbins", "Reconstruction"]).select_dtypes("number").columns
target_col = "Model"

pvals = []
for col in feature_cols:
    groups = [group[col].values for _, group in df_combined.groupby(target_col)]
    _, p = kruskal(*groups)
    pvals.append(p)

# BH correction
reject, pvals_corrected, _, _ = multipletests(pvals, alpha=0.25, method='fdr_bh')

# Assemble results
kw_df = pd.DataFrame({
    "Feature": feature_cols,
    "p_value": pvals,
    "p_adj": pvals_corrected,
    "Keep": reject
}).set_index("Feature")

selected_features_kw = kw_df[kw_df["Keep"]].index.tolist()
print(f"✅ {len(selected_features_kw)} features selected after BH correction (FDR < 0.25)")

✅ 68 features selected after BH correction (FDR < 0.25)


/var/folders/th/p1bpkycd1lx_c6j9tvg1m21c0000gn/T/ipykernel_10801/2030372890.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  groups = [group[col].values for _, group in df_combined.groupby(target_col)]


In [26]:
import numpy as np

# Subset to selected KW features
df_selected = df_combined[selected_features_kw]

# Compute Spearman correlation matrix
corr_matrix = df_selected.corr(method="spearman").abs()

# Mask the diagonal and lower triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

# Identify pairs with correlation above threshold
threshold = 0.9
high_corr_pairs = [
    (corr_matrix.columns[i], corr_matrix.columns[j])
    for i in range(len(corr_matrix.columns))
    for j in range(len(corr_matrix.columns))
    if mask[i, j] and corr_matrix.iloc[i, j] > threshold
]

# Group correlated features
from collections import defaultdict

grouped = defaultdict(set)
for a, b in high_corr_pairs:
    grouped[a].add(a)
    grouped[a].add(b)
    grouped[b].add(a)
    grouped[b].add(b)

# Merge overlapping groups
def merge_groups(groups):
    out = []
    seen = set()
    for g in groups.values():
        if g & seen:
            continue
        queue = list(g)
        cluster = set()
        while queue:
            node = queue.pop()
            if node in cluster:
                continue
            cluster.add(node)
            queue.extend(groups.get(node, []))
        seen |= cluster
        out.append(cluster)
    return out

clusters = merge_groups(grouped)

# Choose 1 feature per group — the one with smallest p_adj
final_features = set(selected_features_kw)
for cluster in clusters:
    if len(cluster) > 1:
        best = kw_df.loc[list(cluster)].sort_values("p_adj").index[0]
        final_features -= (cluster - {best})

selected_features_final = list(final_features)
print(f"✅ {len(selected_features_final)} features remaining after correlation filtering")


✅ 16 features remaining after correlation filtering


### Classification - Model Training

In [27]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import numpy as np

# Inputs
X = df_combined[selected_features_final]
y = df_combined["Model"]

# Use stratified 5-fold CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Classifier setup
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "SVM (RBF kernel)": make_pipeline(StandardScaler(), SVC(kernel="rbf", probability=True, random_state=42))
}

# Run CV
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
    print(f"✅ {name}: {scores.mean():.3f} ± {scores.std():.3f}")


✅ Random Forest: 1.000 ± 0.000
✅ Gradient Boosting: 1.000 ± 0.000
✅ SVM (RBF kernel): 0.968 ± 0.010
